# 05 — Diagnostics & Validation

Aggregate out-of-sample predictions and produce the Phase 1 PASS/FAIL decision.

In [ ]:
import torch, sys
if torch.cuda.is_available():
    print(f"GPU available but diagnostics can run on CPU: {torch.cuda.get_device_name(0)}")
else:
    print("CPU diagnostics mode: no GPU required for notebook 05")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
print('Policy review status:', cfg.get('experiments', {}).get('policy_review', {}).get('status'))

In [ ]:
from google.colab import runtime
import yenibot.experiments as experiments_module
from yenibot.experiments import latest_experiment_run, write_experiment_diagnostics

REPORT_DIR = f'{DRIVE_BASE}/reports'
WRITE_FULL_BUNDLE = False  # Set True only when deep per-profile diagnostic zips are needed.

try:
    print('yenibot.experiments module:', experiments_module.__file__)
    run_dir = latest_experiment_run(CHECKPT_DIR)
    print('Latest experiment run:', run_dir)
    result = write_experiment_diagnostics(
        checkpoint_dir=CHECKPT_DIR,
        config=cfg,
        output_dir=REPORT_DIR,
        write_full_bundles=WRITE_FULL_BUNDLE,
    )
    display(result['comparison'])
    if not result.get('missing_selected_profiles').empty:
        print('Missing selected profiles detected:')
        display(result['missing_selected_profiles'])
    if not result.get('seed_stability').empty:
        display(result['seed_stability'])
    if not result.get('seed_ensemble').empty:
        display(result['seed_ensemble'])
    if not result.get('experiment_selection').empty:
        display(result['experiment_selection'])
    if not result.get('experiment_policy_guard').empty:
        print('Experiment policy guard:')
        display(result['experiment_policy_guard'])
    if not result.get('future_oos_candidate_plan').empty:
        print('Future OOS candidate plan:')
        display(result['future_oos_candidate_plan'])
    if not result.get('performance_gap_analysis').empty:
        print('Performance gap analysis:')
        display(result['performance_gap_analysis'])
    if not result.get('payoff_alignment_summary').empty:
        print('Payoff alignment summary:')
        display(result['payoff_alignment_summary'])
    if not result.get('payoff_policy_robustness_summary').empty:
        print('Payoff policy robustness summary:')
        display(result['payoff_policy_robustness_summary'])
    if not result.get('frozen_policy_monitoring_plan').empty:
        print('Frozen policy monitoring plan:')
        display(result['frozen_policy_monitoring_plan'])
    if not result.get('holdout_evaluation').empty:
        print('Holdout evaluation:')
        display(result['holdout_evaluation'])
    if not result.get('holdout_score_bands').empty:
        print('Holdout score bands:')
        display(result['holdout_score_bands'])
    holdout = result['decision'].get('holdout', {})
    if holdout:
        print('Holdout reservation:', holdout)
    print('Decision:', result['decision'])
    print('Full bundle enabled:', result.get('write_full_bundles'))
    print('Diagnostic zips:')
    for zip_path in result['zip_paths']:
        print(zip_path)
    print('Single download bundle:', result['bundle_zip'])
    print('Latest bundle shortcut:', result['latest_bundle_zip'])
    print('Slim summary bundle:', result.get('slim_bundle_zip'))
    print('Latest slim shortcut:', result.get('latest_slim_bundle_zip'))
    print('Auto review:', result.get('auto_review_path'))
    print('Next actions:', result.get('next_actions_path'))
    print('Phase 2 readiness:', result.get('phase2_readiness_path'))
finally:
    runtime.unassign()
